In [280]:
import numpy as np
import pandas as pd

seed_order = np.array([[[1, 16, 8, 9, 5, 12, 4, 13, 6, 11, 3, 14, 7, 10, 2, 15]]])

poss = np.repeat(seed_order, swap_ix.shape[0]+1, axis=0)
swap_ix = np.array([[ix1, ix2] 
                    for ix1 in range(poss.shape[2]-1) 
                    for ix2 in range(ix1+1, poss.shape[2])])

swap_seed_1 = poss[np.arange(swap_ix.shape[0])+1, 0, swap_ix[:, 0]]
swap_seed_2 = poss[np.arange(swap_ix.shape[0])+1, 0, swap_ix[:, 1]]
swap_labels = np.append([[0, 0]], np.column_stack((swap_seed_1, swap_seed_2)), axis=0)

poss[np.arange(swap_ix.shape[0])+1, 0, swap_ix[:, 0]] = swap_seed_2
poss[np.arange(swap_ix.shape[0])+1, 0, swap_ix[:, 1]] = swap_seed_1

probs = np.ones(poss[:, :, :1].shape)

In [281]:
while poss.shape[2] > 1:
    nbk = int(poss.shape[2] / 2)
    probs = np.prod(np.stack([np.tile(np.repeat((np.flip(poss[:, :, 2*i:2*(i+1)], axis=2) 
                                                 / np.sum(poss[:, :, 2*i:2*(i+1)], axis=2, keepdims=True)), 
                                                2**(nbk-i-1), axis=2), 
                                      (1, 1, 2**i)).reshape(poss.shape[0], -1)
                              for i in range(nbk)],
                             axis=2),
                    axis=2, keepdims=True) * np.repeat(probs, 2**nbk, axis=1)
    poss = np.stack([np.tile(np.repeat(poss[:, :, 2*i:2*(i+1)], 
                                       2**(nbk-i-1), axis=2), 
                             (1, 1, 2**i)).reshape(poss.shape[0], -1)
                     for i in range(nbk)],
                    axis=2)

In [293]:
p2win = np.sum(np.where(poss == 2, probs, 0.), axis=(1, 2))
sortorder = np.argsort(-p2win)
pd.DataFrame({'swapped_1': swap_labels[sortorder, 0],
              'swapped_2': swap_labels[sortorder, 1],
              'win_prob': p2win[sortorder],
              'improvement': (p2win[sortorder] - p2win[0])})

,swapped_1,swapped_2,win_prob,improvement
0,16,3,0.281619,0.065580
1,9,3,0.262324,0.046285
2,8,3,0.260327,0.044287
3,13,3,0.256157,0.040117
4,12,3,0.255542,0.039502
...,...,...,...,...
116,9,2,0.149637,-0.066402
117,12,2,0.149170,-0.066870
118,13,2,0.147519,-0.068520
119,16,2,0.137804,-0.078235


In [294]:
p2win[sortorder][0] - p2win[0]

np.float64(0.06557950370257642)